# 04 - Causal Forest

`econml.grf.CausalForest`: an honest, tree-ensemble CATE estimator, and the
last comparator in this project's portfolio (random reference, Response
LightGBM, T-Learner, X-Learner, Causal Forest).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from src.data import PRIMARY_OUTCOME, TREATMENT_COLUMN, load_parquet
from src.preprocessing import CausalForestCategoricalEncoder
from src.models import fit_causal_forest, predict_causal_forest_tau
from src.evaluation import evaluate_ranking

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
train_frame = load_parquet(PROCESSED_DIR / "train.parquet")
val_frame = load_parquet(PROCESSED_DIR / "validation.parquet")

T_train, T_val = train_frame[TREATMENT_COLUMN], val_frame[TREATMENT_COLUMN]
Y_train, Y_val = train_frame[PRIMARY_OUTCOME], val_frame[PRIMARY_OUTCOME]

## Categorical representation

`CausalForest` has no LightGBM-equivalent native categorical support -- it
consumes a dense numeric matrix. Encoding the categorical tokens as raw
ordinal integers would fabricate a false ordering the forest's splits could
exploit (the same mistake D32 fixes for the raw data).

The representation used here is **frequency-capped top-K one-hot**: for each
categorical feature, keep the top-`K` categories by train-set frequency,
bucket everything else (including categories unseen at train time) into an
explicit `OTHER` level, then one-hot encode. Continuous features pass through
unchanged. `K` is chosen from a resource ladder (32 -> 16 -> 8, picking the
largest that fits available memory/runtime), never by comparing model
performance across `K` values -- unbounded one-hot is infeasible at full data
scale, and target/effect encoding was rejected because it would leak the
outcome into the same matrix the forest uses to place its splits.

In [ ]:
encoder = CausalForestCategoricalEncoder(k=32)
X_train_cf = encoder.fit_transform(train_frame)
X_val_cf = encoder.transform(val_frame)
X_train_cf.shape, X_val_cf.shape

## Fit

`honest=True` and `n_jobs=1` are load-bearing, not defaults left alone:
honesty is required for valid inference, and `n_jobs=1` is required for
determinism (`n_jobs=2`/`-1` were empirically found to produce different
predictions than `n_jobs=1` with an identical `random_state`).

In [ ]:
causal_forest = fit_causal_forest(X_train_cf, T_train, Y_train, seed=42)

## Evaluate

In [ ]:
cf_scores = predict_causal_forest_tau(causal_forest, X_val_cf)
cf_ranking = evaluate_ranking(cf_scores, T_val, Y_val)
print("Causal Forest qini_above_random:", round(cf_ranking.qini_above_random, 6))
cf_ranking.uplift_at_k

In [ ]:
import matplotlib.pyplot as plt

plt.plot(cf_ranking.qini_curve["coverage"], cf_ranking.qini_curve["qini_gain"], label="Causal Forest")
plt.plot([0, 1], [0, cf_ranking.theoretical_random_qini_area * 2], "--", label="Theoretical random")
plt.xlabel("Population coverage")
plt.ylabel("Cumulative incremental conversions")
plt.legend()
plt.title("Causal Forest: Qini curve")

## Conclusions

Summarize `qini_above_random` for all four models here (random reference,
Response LightGBM, T-Learner, X-Learner, Causal Forest) once every notebook
has been run, and compare against `02_baseline_models.ipynb` /
`03_uplift_models.ipynb`'s results.

Any conclusion drawn from this comparison is dataset-specific
(CRITEO-UPLIFTv2.1), implementation-specific (these exact estimators and
hyperparameters), and metric-specific (Qini-above-random on this validation
split) -- not a claim that one estimator is universally best, or that
observed performance is intrinsic to a meta-learner family in general.